In [13]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import pandas as pd
from sqlalchemy import text
from literature_ai.db import ENGINE

## 1. All tables

In [14]:
tables = pd.read_sql(
    """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
      AND table_type = 'BASE TABLE'
    ORDER BY table_schema, table_name
    """,
    ENGINE,
)
tables

,table_schema,table_name
0,processed,abstract_embeddings
1,processed,embedding_runs_metadata
2,processed,processed_abstracts
3,raw,full_papers
4,raw,raw_paper_searches


## 2. Schema per table

In [15]:
for _, row in tables.iterrows():
    schema, table = row['table_schema'], row['table_name']
    cols = pd.read_sql(
        text(
            """
            SELECT column_name, data_type, character_maximum_length,
                   is_nullable, column_default
            FROM information_schema.columns
            WHERE table_schema = :schema AND table_name = :table
            ORDER BY ordinal_position
            """
        ),
        ENGINE,
        params={'schema': schema, 'table': table},
    )
    print(f"\n{'─' * 60}")
    print(f"  {schema}.{table}")
    print(f"{'─' * 60}")
    display(cols)


────────────────────────────────────────────────────────────
  processed.abstract_embeddings
────────────────────────────────────────────────────────────


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,paperId,text,None,NO,None
1,run_id,bigint,None,NO,None
2,processed_at,timestamp with time zone,None,YES,None
3,content_hash,text,None,YES,None
4,embedding_768,USER-DEFINED,None,YES,None
5,embedding_1536,USER-DEFINED,None,YES,None



────────────────────────────────────────────────────────────
  processed.embedding_runs_metadata
────────────────────────────────────────────────────────────


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,run_id,bigint,None,NO,None
1,ran_at,timestamp with time zone,None,NO,now()
2,embedding_model,text,None,NO,None
3,embedding_version,text,None,YES,None
4,n_dim,integer,None,NO,None
5,user_tags,jsonb,None,NO,'{}'::jsonb
6,source,text,None,NO,None



────────────────────────────────────────────────────────────
  processed.processed_abstracts
────────────────────────────────────────────────────────────


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,paperId,text,None,NO,None
1,title,text,None,YES,None
2,abstract,text,None,YES,None
3,abstract_length,integer,None,YES,None
4,word_count,integer,None,YES,None
5,has_formula,boolean,None,YES,None
6,language,text,None,YES,None
7,content_hash,text,None,YES,None
8,processed_at,timestamp with time zone,None,YES,None



────────────────────────────────────────────────────────────
  raw.full_papers
────────────────────────────────────────────────────────────


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,paperId,text,None,NO,None
1,full_text,text,None,YES,None
2,pdf_url,text,None,YES,None
3,collected_at,timestamp with time zone,None,NO,None



────────────────────────────────────────────────────────────
  raw.raw_paper_searches
────────────────────────────────────────────────────────────


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,paperId,text,None,NO,None
1,title,text,None,YES,None
2,abstract,text,None,YES,None
3,venue,text,None,YES,None
4,year,integer,None,YES,None
5,citationCount,integer,None,YES,None
6,influentialCitationCount,integer,None,YES,None
7,fieldsOfStudy,jsonb,None,YES,None
8,isOpenAccess,boolean,None,YES,None
9,publicationTypes,jsonb,None,YES,None


## 3. Sample rows

In [16]:
for _, row in tables.iterrows():
    schema, table = row['table_schema'], row['table_name']
    sample = pd.read_sql(
        f'SELECT * FROM "{schema}"."{table}" LIMIT 5',
        ENGINE,
    )
    print(f"\n{'─' * 60}")
    print(f"  {schema}.{table}  ({len(sample)} rows shown)")
    print(f"{'─' * 60}")
    display(sample)


────────────────────────────────────────────────────────────
  processed.abstract_embeddings  (5 rows shown)
────────────────────────────────────────────────────────────


,paperId,run_id,processed_at,content_hash,embedding_768,embedding_1536
0,2c03df8b48bf3fa39054345bafabfeff15bfd11d,1,2026-07-12 20:28:34.524705+00:00,b6e91e48dcd2d7b65c162f1f13c76f122711edc2177b94...,"[0.2029713,0.6437859,-0.3180334,0.32343253,-0....",None
1,dc32a984b651256a8ec282be52310e6bd33d9815,1,2026-07-12 20:28:34.524705+00:00,8c876b291245a21818d93f169b084b03047fb5f285b0f9...,"[0.4027466,0.6113165,-0.1346894,-0.2396962,-0....",None
2,5582bebed97947a41e3ddd9bd1f284b73f1648c2,1,2026-07-12 20:28:34.524705+00:00,f8af338ebfcbc66cc0d243c75d4a00e3a2c3e54de47a17...,"[0.014119079,0.79693395,-0.36396796,-0.3719218...",None
3,4c75b748911ddcd888c5122f7672f69caa5d661f,1,2026-07-12 20:28:34.524705+00:00,9f067eb9eb59b7c48d8017f1b88da2ef44ed78158d6b5f...,"[-0.32550272,0.5053523,-0.24488288,-0.37329957...",None
4,cab372bc3824780cce20d9dd1c22d4df39ed081a,1,2026-07-12 20:28:34.524705+00:00,c6ac5a5ebbfc98d145b8b4c53f1d6e61e021a18a1c9c56...,"[0.3214283,0.43594941,-0.3490806,-0.2749592,-0...",None



────────────────────────────────────────────────────────────
  processed.embedding_runs_metadata  (2 rows shown)
────────────────────────────────────────────────────────────


,run_id,ran_at,embedding_model,embedding_version,n_dim,user_tags,source
0,1,2026-07-12 20:07:55.810141+00:00,specter_v2,None,768,{},collect
1,2,2026-07-12 20:08:06.227625+00:00,text-embedding-3-small,None,1536,{},generate



────────────────────────────────────────────────────────────
  processed.processed_abstracts  (5 rows shown)
────────────────────────────────────────────────────────────


,paperId,title,abstract,abstract_length,word_count,has_formula,language,content_hash,processed_at
0,2c03df8b48bf3fa39054345bafabfeff15bfd11d,Deep Residual Learning for Image Recognition,Deeper neural networks are more difficult to t...,1293,201,False,en,b6e91e48dcd2d7b65c162f1f13c76f122711edc2177b94...,2026-07-12 20:06:04.735380+00:00
1,dc32a984b651256a8ec282be52310e6bd33d9815,Highly accurate protein structure prediction w...,"Proteins are essential to life, and understand...",1841,251,False,en,8c876b291245a21818d93f169b084b03047fb5f285b0f9...,2026-07-12 20:06:04.735380+00:00
2,5582bebed97947a41e3ddd9bd1f284b73f1648c2,Grad-CAM: Visual Explanations from Deep Networ...,We propose a technique for producing ‘visual e...,2537,344,False,en,f8af338ebfcbc66cc0d243c75d4a00e3a2c3e54de47a17...,2026-07-12 20:06:04.735380+00:00
3,4c75b748911ddcd888c5122f7672f69caa5d661f,Statistical Learning Theory,"A machine learning system, in general, learns ...",738,101,False,en,9f067eb9eb59b7c48d8017f1b88da2ef44ed78158d6b5f...,2026-07-12 20:06:04.735380+00:00
4,cab372bc3824780cce20d9dd1c22d4df39ed081a,DeepLab: Semantic Image Segmentation with Deep...,In this work we address the task of semantic i...,1712,248,False,en,065b65aee4717a95102391398502db3ff6153c64a24983...,2026-07-12 20:06:04.735380+00:00



────────────────────────────────────────────────────────────
  raw.full_papers  (0 rows shown)
────────────────────────────────────────────────────────────


,paperId,full_text,pdf_url,collected_at



────────────────────────────────────────────────────────────
  raw.raw_paper_searches  (5 rows shown)
────────────────────────────────────────────────────────────


,paperId,title,abstract,venue,year,citationCount,influentialCitationCount,fieldsOfStudy,isOpenAccess,publicationTypes,url,status,ArXiV,DBLP,MAG,DOI,collected_at,last_updated
0,2c03df8b48bf3fa39054345bafabfeff15bfd11d,Deep Residual Learning for Image Recognition,Deeper neural networks are more difficult to t...,Computer Vision and Pattern Recognition,2015,233194,32493,[Computer Science],True,"[JournalArticle, Conference]",https://repositorio.unal.edu.co/bitstream/unal...,GREEN,None,conf/cvpr/HeZRS16,2949650786,10.1109/cvpr.2016.90,2026-07-12 09:35:44.542599+00:00,2026-07-12 09:35:44.542599+00:00
1,dc32a984b651256a8ec282be52310e6bd33d9815,Highly accurate protein structure prediction w...,"Proteins are essential to life, and understand...",Nature,2021,37256,3859,[Medicine],True,[JournalArticle],https://www.nature.com/articles/s41586-021-038...,HYBRID,None,None,None,10.1038/s41586-021-03819-2,2026-07-12 09:35:44.542599+00:00,2026-07-12 09:35:44.542599+00:00
2,5582bebed97947a41e3ddd9bd1f284b73f1648c2,Grad-CAM: Visual Explanations from Deep Networ...,We propose a technique for producing ‘visual e...,International Journal of Computer Vision,2016,27947,2461,[Computer Science],True,[JournalArticle],https://arxiv.org/pdf/1610.02391,GREEN,None,conf/iccv/SelvarajuCDVPB17,2962858109,10.1007/s11263-019-01228-7,2026-07-12 09:35:44.542599+00:00,2026-07-12 09:35:44.542599+00:00
3,4c75b748911ddcd888c5122f7672f69caa5d661f,Statistical Learning Theory,"A machine learning system, in general, learns ...",Technometrics,2021,22087,2594,[Computer Science],True,[JournalArticle],https://pure.mpg.de/pubman/item/item_1792389_3...,CLOSED,None,journals/technometrics/Wu99,1926872911,10.1080/00401706.1999.10485951,2026-07-12 09:35:44.542599+00:00,2026-07-12 09:35:44.542599+00:00
4,cab372bc3824780cce20d9dd1c22d4df39ed081a,DeepLab: Semantic Image Segmentation with Deep...,In this work we address the task of semantic i...,IEEE Transactions on Pattern Analysis and Mach...,2016,21206,1849,"[Computer Science, Medicine]",True,[JournalArticle],https://arxiv.org/pdf/1606.00915,GREEN,None,journals/pami/ChenPKMY18,2952865063,10.1109/TPAMI.2017.2699184,2026-07-12 09:35:44.542599+00:00,2026-07-12 09:35:44.542599+00:00
